# Предсказание рейтинга игрока

Полный пайплайн: модель + голова рейтинга. Голова получает **эмбеддинги с выхода трансформера** (после attention). Указать путь до модели и головы, задать игрока (имя/id), позицию, команду и статистики (39 признаков) — получить предсказанный overall рейтинг.

In [1]:
import sys
from pathlib import Path
import pickle

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from safetensors.torch import load_file as load_safetensors

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: mps


## Пути к модели и голове

In [2]:
# Укажите пути (можно менять)
MODEL_DIR = ROOT / "notebooks" / "mpp_mini_output" / "final_model"   # папка с model.safetensors
HEAD_PATH = ROOT / "notebooks" / "mpp_mini_output" / "rating_head.pt"  # веса головы регрессии
METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"  # player_name2id, team_name2id

In [3]:
# Загрузка словарей
from pathlib import Path
import pickle

def load_vocab(metadata_dir: Path):
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n_players = len(player_name2id)
    n_teams = len(team_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n_players + 1,
        "team_pad_token_id": n_teams,
    }

vocab = load_vocab(METADATA_DIR)
player_name2id = vocab["player_name2id"]
team_name2id = vocab["team_name2id"]
player_pad_token_id = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1
print("Игроков в словаре:", len(player_name2id), "| Команд:", len(team_name2id))

Игроков в словаре: 6393 | Команд: 243


In [4]:
from models.encoder import PlayerEncoder
from models.heads import RegressionHead
from data.preprocessing import STAT_COLUMNS, FORM_STATS_SIZE

embed_size = 128
encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=player_pad_token_id,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state = load_safetensors(MODEL_DIR / "model.safetensors")
encoder_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
encoder.load_state_dict(encoder_state, strict=True)

head = RegressionHead(embed_size=embed_size, output_dim=1, hidden_dim=128, pool="per_sequence")
head.load_state_dict(torch.load(HEAD_PATH, map_location="cpu", weights_only=True))

encoder = encoder.to(device).eval()
head = head.to(device).eval()
print("Модель и голова загружены.")

Модель и голова загружены.


## Ввод данных игрока и предсказание

- **player_name** или **player_id**: имя (из словаря) или числовой id игрока.  
- **position_id**: 1–25 (StatsBomb).  
- **team_name** или **team_id**: опционально.  
- **stats**: словарь или список из 39 чисел (порядок STAT_COLUMNS); неуказанные = 0.  
- Голова получает эмбеддинги **с выхода трансформера** (после attention), как при обучении.

In [5]:
import numpy as np

def stats_to_vector(stats) -> np.ndarray:
    """Преобразовать stats в вектор длины 39 в порядке STAT_COLUMNS."""
    if isinstance(stats, (list, np.ndarray)):
        arr = np.asarray(stats, dtype=np.float32)
        if arr.size != FORM_STATS_SIZE:
            raise ValueError(f"Ожидается {FORM_STATS_SIZE} статов, получено {arr.size}")
        return arr
    # dict: по именам колонок
    vec = np.zeros(FORM_STATS_SIZE, dtype=np.float32)
    for i, name in enumerate(STAT_COLUMNS):
        if name in stats:
            vec[i] = float(stats[name])
    return vec

def predict_rating(
    player_name: str | None = None,
    player_id: int | None = None,
    position_id: int = 1,
    team_name: str | None = None,
    team_id: int | None = None,
    stats: dict | list | None = None,
) -> float:
    """
    Предсказать рейтинг одного игрока. Голова берёт эмбеддинги с выхода трансформера.
    Обязательно: player_name или player_id. Остальное опционально (position_id,
    team_id, stats — по умолчанию 0).
    """
    if player_id is None:
        if player_name is None:
            raise ValueError("Укажите player_name или player_id")
        player_id = player_name2id.get(player_name, player_pad_token_id - 1)
    if team_id is None and team_name is not None:
        team_id = team_name2id.get(team_name, 0)
    if team_id is None:
        team_id = 0
    if stats is None:
        stats = {}
    form_stats = stats_to_vector(stats)

    pos_id = position_id - 1 if 1 <= position_id <= 25 else 0
    input_ids = torch.tensor([[player_id]], dtype=torch.long, device=device)
    position_ids = torch.tensor([[pos_id]], dtype=torch.long, device=device)
    team_ids = torch.tensor([[team_id]], dtype=torch.long, device=device)
    form_stats_t = torch.tensor(form_stats.reshape(1, 1, -1), dtype=torch.float32, device=device)
    attention_mask = torch.ones(1, 1, dtype=torch.long, device=device)

    with torch.no_grad():
        enc_out, _ = encoder(input_ids, position_ids, team_ids, form_stats_t, attention_mask)
        rating = head(enc_out, attention_mask)

    return float(rating.squeeze().cpu().item())

## Пример: задать статистики и получить рейтинг

In [6]:
# Пример 1: по имени игрока и статам (эмбеддинги после трансформера)
player_name = "Brenden Aaronson"
position_id = 10
stats_dict = {"pass_total": 50, "pass_goal_assist": 1, "shot_total": 4, "shot_goal": 1}

rating = predict_rating(player_name=player_name, position_id=position_id, stats=stats_dict)
print(f"Предсказанный рейтинг: {rating:.2f}")

Предсказанный рейтинг: 64.99


In [7]:
# Пример 2: по player_id и списку из 39 статов
player_id = 0
stats_list = [0.0] * FORM_STATS_SIZE
stats_list[0], stats_list[3], stats_list[10], stats_list[19] = 60.0, 2.0, 3.0, 1.0

rating2 = predict_rating(player_id=player_id, position_id=6, stats=stats_list)
print(f"Предсказанный рейтинг: {rating2:.2f}")

Предсказанный рейтинг: 71.76


In [8]:
# Список имён статов (порядок для списка из 39 чисел):
for i, name in enumerate(STAT_COLUMNS):
    print(f"{i}: {name}")

0: pass_total
1: pass_cross
2: pass_cut_back
3: pass_shot_assist
4: pass_goal_assist
5: pass_no_touch
6: pass_interception
7: pass_incomplete
8: pass_offside
9: pass_through_ball
10: shot_total
11: shot_statsbomb_xg
12: shot_corner
13: shot_free_kick
14: shot_open_play
15: shot_penalty
16: shot_saved
17: shot_off_target
18: shot_blocked
19: shot_goal
20: interception_total
21: interception_won
22: interception_lost
23: block_total
24: clearance_total
25: ball_recovery_total
26: dribble_total
27: dribble_complete
28: dribble_incomplete
29: foul_won_total
30: foul_won_penalty
31: foul_committed_total
32: foul_committed_penalty
33: foul_committed_yellow_card
34: foul_committed_red_card
35: goalkeeper_goal_conceded
36: goalkeeper_save
37: goalkeeper_shot_faced
38: counterpress_total
